In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import lightning.pytorch as pl
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger


In [4]:
DATA_PATH = Path(r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet")

df = pd.read_parquet(DATA_PATH).copy()

# Ensure time is datetime and sorted
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df.sort_values("time").reset_index(drop=True)

print(df.head())
print("Shape:", df.shape)


                       time  volume    mid_o    mid_h    mid_l    mid_c  \
0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   

     bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787  
1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038  
2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159  
3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196  
4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245  
Shape: (61473, 14)


In [5]:
import sys
from pathlib import Path

project_root = Path(r"C:\Users\admin\Desktop\Forex_algo\code")
sys.path.append(str(project_root))

from technicals.indicators import BollingerBands, ATR, KeltnerChannels, RSI, MACD

# Apply your existing indicators (adapt if you already did this elsewhere)
df = BollingerBands(df)
df = ATR(df, n=14)
df = KeltnerChannels(df, n_ema=20, n_atr=10)
df = RSI(df, n=14)
df = MACD(df, n_slow=26, n_fast=12, n_signal=9)

# Extra EMAs using pandas
df["ema_5"]   = df["mid_c"].ewm(span=5,  min_periods=5).mean()
df["ema_20"]  = df["mid_c"].ewm(span=20, min_periods=20).mean()
df["ema_50"]  = df["mid_c"].ewm(span=50, min_periods=50).mean()
df["ema_200"] = df["mid_c"].ewm(span=200, min_periods=200).mean()


In [6]:
HORIZON = 4          # predict 4h ahead; you can try 1, 4, 6 etc.
UP_THRESH = 0.0005   # ~5 pips when price ~1.0
DOWN_THRESH = -0.0005

def make_labels(df, horizon=HORIZON, up_th=UP_THRESH, down_th=DOWN_THRESH):
    df = df.copy()
    # future log return
    df["future_ret"] = np.log(df["mid_c"].shift(-horizon) / df["mid_c"])

    # init with 'unknown' = 2
    df["label"] = 2

    df.loc[df["future_ret"] >= up_th, "label"] = 0  # up
    df.loc[df["future_ret"] <= down_th, "label"] = 1  # down

    # drop tail rows where future_ret is NaN
    df = df.dropna(subset=["future_ret"]).reset_index(drop=True)
    return df

df_lab = make_labels(df)

df_lab["label"].value_counts(normalize=True)


label
0    0.340058
1    0.335210
2    0.324733
Name: proportion, dtype: float64

In [7]:
FEATURE_COLS = [
    "mid_o", "mid_h", "mid_l", "mid_c", "volume",
    "RSI_14",      # from your RSI()
    "MACD", "SIGNAL", "HIST",
    "BB_LW", "BB_MA", "BB_UP",
    "ATR_14",
    "ema_5", "ema_20", "ema_50", "ema_200",
]

# Keep only rows with all features valid
df_lab = df_lab.dropna(subset=FEATURE_COLS + ["label"]).reset_index(drop=True)

WINDOW_SIZE = 48  # last 48 hours as input sequence
N_CLASSES = 3

print(df_lab[FEATURE_COLS + ["label"]].head())


     mid_o    mid_h    mid_l    mid_c  volume     RSI_14      MACD    SIGNAL  \
0  1.08736  1.08764  1.08595  1.08652    1298  36.239350 -0.000565 -0.000376   
1  1.08654  1.08846  1.08636  1.08846    1450  47.615598 -0.000518 -0.000404   
2  1.08844  1.08890  1.08702  1.08724    1626  42.482293 -0.000572 -0.000438   
3  1.08728  1.08834  1.08662  1.08730    1376  42.808846 -0.000603 -0.000471   
4  1.08728  1.08732  1.08599  1.08630     941  38.849941 -0.000700 -0.000517   

       HIST     BB_LW     BB_MA     BB_UP    ATR_14     ema_5    ema_20  \
0 -0.000189  1.087360  1.088880  1.090399  0.001128  1.087710  1.088764   
1 -0.000113  1.087218  1.088804  1.090390  0.001199  1.087960  1.088735   
2 -0.000134  1.087091  1.088718  1.090344  0.001283  1.087720  1.088593   
3 -0.000132  1.086948  1.088604  1.090259  0.001351  1.087580  1.088470   
4 -0.000184  1.086639  1.088526  1.090413  0.001391  1.087153  1.088263   

     ema_50   ema_200  label  
0  1.089161  1.088334      2  
1  1.0

In [8]:
n = len(df_lab)
train_end = int(n * 0.8)

train_df = df_lab.iloc[:train_end].reset_index(drop=True)
val_df   = df_lab.iloc[train_end:].reset_index(drop=True)

print("Train size:", len(train_df), "Val size:", len(val_df))


Train size: 49016 Val size: 12254


In [9]:
class SeqDataset(Dataset):
    def __init__(self, df, feature_cols, window_size):
        self.df = df
        self.feature_cols = feature_cols
        self.window_size = window_size

        self.X = []
        self.y = []

        data = df[feature_cols].values.astype(np.float32)
        labels = df["label"].values.astype(np.int64)

        # sequences end at index i, start at i-window_size+1
        for i in range(window_size - 1, len(df)):
            x_seq = data[i - window_size + 1 : i + 1]   # [window_size, n_features]
            y_val = labels[i]
            self.X.append(x_seq)
            self.y.append(y_val)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])         # [T, F]
        y = torch.tensor(self.y[idx])             # scalar
        return x, y

train_ds = SeqDataset(train_df, FEATURE_COLS, WINDOW_SIZE)
val_ds   = SeqDataset(val_df,   FEATURE_COLS, WINDOW_SIZE)

print("Train samples:", len(train_ds), "Val samples:", len(val_ds))


Train samples: 48969 Val samples: 12207


In [17]:
BATCH_SIZE = 512

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # <-- important on Windows
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,      # <-- important on Windows
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train batches: 96
Val batches: 24


In [18]:
class ConvBiLSTMClassifier(pl.LightningModule):
    def __init__(
        self,
        n_features,
        n_classes=3,
        window_size=48,
        lr=1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.n_features = n_features
        self.n_classes = n_classes
        self.window_size = window_size
        self.lr = lr

        # Conv1d expects [B, C, T]
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=64,        out_channels=128, kernel_size=3, padding=1)

        # LSTM expects [B, T, C] – make it bidirectional
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(0.3)
        # bidirectional → hidden size * 2
        self.fc = nn.Linear(64 * 2, n_classes)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        # x: [B, T, F]
        x = x.permute(0, 2, 1)              # [B, F, T]
        x = torch.relu(self.conv1(x))       # [B, 64, T]
        x = torch.relu(self.conv2(x))       # [B, 128, T]

        x = x.permute(0, 2, 1)              # [B, T, 128]
        out, _ = self.lstm(x)               # [B, T, 128] (64 * 2)

        h_last = out[:, -1, :]              # [B, 128]
        h_last = self.dropout(h_last)
        logits = self.fc(h_last)            # [B, n_classes]

        return logits

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, factor=0.5, patience=3, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }


In [19]:
# 1) Use Tensor Cores properly
torch.set_float32_matmul_precision("medium")

# 2) Reproducibility
pl.seed_everything(42)

# 3) Device setup
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

# 4) Callbacks
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_h1_convbilstm-{epoch:02d}-{val_loss:.4f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,      # a bit more patience for deeper model
    min_delta=1e-4,
)

# 5) Logger
logger = CSVLogger(
    save_dir="logs",
    name="eurusd_h1_convbilstm",
)

# 6) Model (use the updated ConvBiLSTMClassifier)
model = ConvBiLSTMClassifier(
    n_features=len(FEATURE_COLS),
    n_classes=N_CLASSES,
    window_size=WINDOW_SIZE,
    lr=1e-3,
)

# 7) Trainer
trainer = Trainer(
    max_epochs=80,             # early stopping will stop earlier if needed
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=0.5,
    callbacks=[checkpoint_cb, earlystop_cb],
    logger=logger,
    log_every_n_steps=20,
    num_sanity_val_steps=0,    # skip sanity check (Windows + num_workers=0)
)

# 8) Train
trainer.fit(model, train_loader, val_loader)

print("Best checkpoint:", checkpoint_cb.best_model_path)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params | Mode 
-----------------------------------------------------
0 | conv1   | Conv1d           | 3.3 K  | train
1 | conv2   | Conv1d           | 24.7 K | train
2 | lstm    | LSTM             | 99.3 K | train
3 | dropout | Dropout          | 0      | train
4 | fc      | Linear           | 387    | train
5 | loss_fn | CrossEntropyLoss | 0      | train
-----------------------------------------------------
127 K     Trainable params
0         Non-trainable params
127 K     Total params
0.511     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Epoch 18: 100%|██████████| 96/96 [00:01<00:00, 73.30it/s, v_num=1, train_loss=1.070, train_acc=0.395, val_loss=1.100, val_acc=0.354]
Best checkpoint: logs\eurusd_h1_convbilstm\version_1\checkpoints\eurusd_h1_convbilstm-epoch=08-val_loss=1.0942.ckpt


In [20]:
print("Train label distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nVal label distribution:")
print(val_df["label"].value_counts(normalize=True))


Train label distribution:
label
0    0.341358
1    0.338767
2    0.319875
Name: proportion, dtype: float64

Val label distribution:
label
2    0.345438
0    0.334177
1    0.320385
Name: proportion, dtype: float64
